# Domain – Chemical Industry

## Focus
Classify chemical

## Business Challenge / Requirement
**FuPont** is a leading chemical company across the globe. The company is on a **CSR (Corporate Social Responsibility)** mission. It wants to identify biodegradable products based on a study of the relationships between chemical structure and biodegradation of molecules.

You as an **ML Expert** have to create an ML model to classify the chemical structure as:
* **'Ready BioDegradable' (RB)** vs 
* **'Not Ready Biodegradable' (NRB)**

## Key Issues
* Data has lots of attributes and classification could be tricky.

## Considerations
* **NONE**

## Data Volume
* **Approx 1,055 records** – file `bio-degradabale-data.csv`

## Fields in Data
* Details in `.ipynb` notebook

## Additional Information
* **NA**

## Business Benefits
Research can lead FuPont to create truly unique **Biodegradable packaging material**. This could lead to massive profits in the future.

In [22]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [23]:
df = pd.read_csv('./files/bio-degradabale-data.csv',sep=';',header=None)
display(df.head())

,0,1,2,3,4,5,6,7,8,9,...,32,33,34,35,36,37,38,39,40,41
0,3.919,2.6909,0,0,0,0,0,31.4,2,0,...,0,0,0,2.949,1.591,0,7.253,0,0,RB
1,4.170,2.1144,0,0,0,0,0,30.8,1,1,...,0,0,0,3.315,1.967,0,7.257,0,0,RB
2,3.932,3.2512,0,0,0,0,0,26.7,2,4,...,0,0,1,3.076,2.417,0,7.601,0,0,RB
3,3.000,2.7098,0,0,0,0,0,20.0,0,2,...,0,0,1,3.046,5.000,0,6.690,0,0,RB
4,4.236,3.3944,0,0,0,0,0,29.4,2,4,...,0,0,0,3.351,2.405,0,8.003,0,0,RB


In [24]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [25]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [26]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_preds = rf_model.predict(X_test_scaled)

In [27]:
from xgboost import XGBClassifier
 
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train)
xgb_preds = xgb_model.predict(X_test_scaled)

d:\repositories\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:17:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [28]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

def evaluate_model(name, y_true, y_pred, model, X_test_data):
    y_prob = model.predict_proba(X_test_data)[:, 1]
    print(f"=== {name} Performance ===")
    print(classification_report(y_true, y_pred, target_names=le.classes_))
    print(f"ROC-AUC Score: {roc_auc_score(y_true, y_prob):.4f}\n")

evaluate_model("Random Forest", y_test, rf_preds, rf_model, X_test_scaled)
evaluate_model("XGBoost", y_test, xgb_preds, xgb_model, X_test_scaled)

=== Random Forest Performance ===
              precision    recall  f1-score   support

         NRB       0.91      0.91      0.91       140
          RB       0.82      0.83      0.83        71

    accuracy                           0.88       211
   macro avg       0.87      0.87      0.87       211
weighted avg       0.88      0.88      0.88       211

ROC-AUC Score: 0.9396

=== XGBoost Performance ===
              precision    recall  f1-score   support

         NRB       0.87      0.94      0.90       140
          RB       0.85      0.72      0.78        71

    accuracy                           0.86       211
   macro avg       0.86      0.83      0.84       211
weighted avg       0.86      0.86      0.86       211

ROC-AUC Score: 0.9241

